# SmartFactory XAI: 원본 모델/배열 재평가 및 동일 코드 재학습
원본 검증의 문제를 수정하지 않는다. 방법 재현과 독립 검증은 구분한다.

In [1]:
from pathlib import Path
import sys, json, hashlib, copy, numpy as np, pandas as pd, torch, sklearn
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score, average_precision_score, precision_recall_curve
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
root=Path.cwd()
if not (root/'data').exists():root=root.parent
assets=root/'data/external_reproduction/smartfactory_xai_dea2c02'
out=root/'reports/tables'; out.mkdir(exist_ok=True,parents=True)
assert sys.version_info[:2]==(3,11)
assert torch.__version__.split('+')[0]=='2.3.1' and np.__version__=='1.26.4' and sklearn.__version__=='1.2.2'
print('VERSIONS',sys.version,torch.__version__,np.__version__,sklearn.__version__,flush=True)
cols=['Injection_Time','Filling_Time','Plasticizing_Time','Cycle_Time','Clamp_Close_Time','Cushion_Position','Plasticizing_Position','Clamp_Open_Position','Max_Injection_Speed','Max_Screw_RPM','Average_Screw_RPM','Max_Injection_Pressure','Max_Switch_Over_Pressure','Max_Back_Pressure','Average_Back_Pressure']+['Barrel_Temperature_'+str(k) for k in range(1,7)]+['Hopper_Temperature','Mold_Temperature_3','Mold_Temperature_4']
class Autoencoder(nn.Module):
 def __init__(self,input_dim=24):
  super().__init__()
  self.encoder=nn.Sequential(nn.Linear(input_dim,16),nn.BatchNorm1d(16),nn.ReLU(),nn.Linear(16,8),nn.ReLU())
  self.decoder=nn.Sequential(nn.Linear(8,16),nn.BatchNorm1d(16),nn.ReLU(),nn.Linear(16,input_dim))
 def forward(self,x):return self.decoder(self.encoder(x))
def errors(model,x):
 model.eval();r=[]
 for i in range(0,len(x),2048):
  xb=torch.FloatTensor(x[i:i+2048])
  with torch.no_grad():r.append(torch.mean((xb-model(xb))**2,dim=1).numpy())
 return np.concatenate(r)
def threshold(model,x,y):
 e=errors(model,x);en=e[y==0]
 a=float(np.percentile(en,99));p,r,c=precision_recall_curve(y,e);f=2*p*r/(p+r+1e-9);k=np.argmax(f[:-1]);b=float(c[k])
 return (a+b)/2,e,a,b
def train_original(model,x,v):
 torch.manual_seed(42);opt=torch.optim.Adam(model.parameters(),lr=.001);criterion=nn.MSELoss()
 xt=torch.FloatTensor(x);xv=torch.FloatTensor(v);loader=DataLoader(TensorDataset(xt,xt),batch_size=256,shuffle=True)
 best=float('inf');patience=0;history=[]
 for epoch in range(1,101):
  model.train();losses=[]
  for xb,_ in loader:
   opt.zero_grad();loss=criterion(model(xb),xb);loss.backward();opt.step();losses.append(loss.item())
  model.eval()
  with torch.no_grad():vl=criterion(model(xv),xv).item()
  history.append(dict(epoch=epoch,train_loss=float(np.mean(losses)),val_loss=vl))
  if epoch%10==0:print('epoch',epoch,'val',vl,flush=True)
  if vl<best-1e-6:
   best=vl;patience=0;state={k:v.clone() for k,v in model.state_dict().items()}
  else:
   patience+=1
   if patience>=10:break
 model.load_state_dict(state)
 return history
rows=[];preds=[];matches=[]
def record(version,kind,y,score,cut):
 pred=(score>=cut).astype(int);tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
 row=dict(version=version,method=kind,n=len(y),bad=int(y.sum()),TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),F1=f1_score(y,pred,zero_division=0),precision=precision_score(y,pred,zero_division=0),recall=recall_score(y,pred,zero_division=0),AP=average_precision_score(y,score),ROC_AUC=roc_auc_score(y,score),threshold=float(cut))
 rows.append(row)
 for i,(actual,s,p) in enumerate(zip(y,score,pred)):preds.append(dict(version=version,method=kind,index=i,actual=int(actual),score=float(s),prediction=int(p)))
 print('RESULT',row,flush=True)
def prepare_local(version):
 files=['supervised_label_cn7.csv'] if version=='v2_cn7_single' else ['supervised_label_cn7.csv','moldset_labeled_cn7.csv','moldset_labeled_rg3.csv']
 frames=[pd.read_csv(root/'data/original'/f,index_col=0)[cols+['PassOrFail']] for f in files]
 df=pd.concat(frames,ignore_index=True);normal=df[df.PassOrFail==0][cols];bad=df[df.PassOrFail==1][cols]
 tr,v=train_test_split(normal,test_size=.2,random_state=42);sc=StandardScaler().fit(tr)
 xx=sc.transform(tr).astype(np.float32);vv=np.concatenate([sc.transform(v).astype(np.float32),sc.transform(bad).astype(np.float32)])
 yy=np.array([0]*len(v)+[1]*len(bad))
 return xx,vv,yy,files
saved={}
for version,suffix,weight,threshold_file in [('v1_combined','','models__v1_backup_autoencoder.pt','models__v1_backup_threshold.json'),('v2_cn7_single','_v2','models__autoencoder_v2_single.pt','models__threshold_v2.json')]:
 x=np.load(assets/('models__X_train'+suffix+'.npy'),allow_pickle=False)
 v=np.load(assets/('models__X_val'+suffix+'.npy'),allow_pickle=False)
 y=np.load(assets/('models__y_val'+suffix+'.npy'),allow_pickle=False)
 xx,vv,yy,files=prepare_local(version)
 eq=xx.shape==x.shape and vv.shape==v.shape and np.array_equal(yy,y)
 delta=float(max(np.max(np.abs(xx-x)),np.max(np.abs(vv-v)))) if eq else None
 matches.append(dict(version=version,local_files='|'.join(files),shape_labels_match=eq,max_absolute_difference=delta,exact_arrays=bool(eq and np.array_equal(xx,x) and np.array_equal(vv,v))))
 print('LOCAL ARRAY MATCH',matches[-1],flush=True)
 model=Autoencoder();model.load_state_dict(torch.load(assets/weight,map_location='cpu',weights_only=True))
 cut=float(json.loads((assets/threshold_file).read_text())['value'])
 e=errors(model,v);record(version,'published_weights_published_threshold',y,e,cut)
 recomputed,e,pc,fc=threshold(model,v,y);record(version,'published_weights_original_threshold_recomputed',y,e,recomputed)
 saved[version]=(x,v,y,model,e,cut)
 # Source 07_train_baselines.py settings unchanged; do not deserialize remote pickle.
 models=[IsolationForest(n_estimators=200,contamination=.01,random_state=42,n_jobs=-1),OneClassSVM(nu=.01,kernel='rbf',gamma='auto'),LocalOutlierFactor(n_neighbors=20,contamination=.01,novelty=True,n_jobs=-1)]
 votes=[e>=cut]
 for name,m in zip(['IF','OCSVM','LOF'],models):
  m.fit(x);s=-m.score_samples(v);c=float(np.percentile(s[y==0],99));votes.append(s>=c);record(version,name,y,s,c)
 vote=np.column_stack(votes).sum(1)
 for k in range(1,5):record(version,'consensus_'+str(k),y,vote,k)
# Re-train using original 01_train/ trainer settings on exactly the deposited arrays.
for version in ['v2_cn7_single','v1_combined']:
 x,v,y,_,_,_=saved[version]
 np.random.seed(42);torch.manual_seed(42);model=Autoencoder()
 hist=train_original(model,x,v[y==0]);cut,e,pc,fc=threshold(model,v,y)
 record(version,'original_training_repeated',y,e,cut)
 torch.save(model.state_dict(),assets/('local_retrained_'+version+'.pt'))
 pd.DataFrame(hist).to_csv(out/('smartfactory_original_history_'+version+'.csv'),index=False)
pd.DataFrame(rows).to_csv(out/'smartfactory_original_reproduction_metrics.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(preds).to_csv(out/'smartfactory_original_reproduction_predictions.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(matches).to_csv(out/'smartfactory_original_array_verification.csv',index=False,encoding='utf-8-sig')
manifest=[dict(file=p.name,sha256=hashlib.sha256(p.read_bytes()).hexdigest(),bytes=p.stat().st_size) for p in assets.iterdir() if p.is_file()]
pd.DataFrame(manifest).to_csv(out/'smartfactory_original_assets_sha256.csv',index=False)
report='# SmartFactory XAI 원본 재현\n\n'
report+='Source commit dea2c02fdefeac7bad9ff207187726fd056094cc. https://github.com/tkddnjs-dlqslek/smartfactory-xai/tree/dea2c02fdefeac7bad9ff207187726fd056094cc\n\n'
report+='Python3.11, PyTorch2.3.1, NumPy1.26.4, pandas2.1.4, sklearn1.2.2 전용 환경. 원본24센서·AE24→16→8→16→24 및 BatchNorm, MSE, Adam.001, batch256, seed42,100epoch최대,patience10,min_delta1e-6 그대로 사용.\n\n'
report+='원본 배열·가중치 추론 재평가와 원본 학습 코드의 재학습을 구분. 원격 pickle 모델은 실행하지 않고 공개 state_dict만 weights_only=True로 읽음. IF200트리·contamination.01,OCSVM gamma auto/nu.01,LOF20/novelty=True 설정도 그대로. 임계값은 원본처럼 평가 정상99%와 평가 전체 F1최적값 평균; baselines는 평가 정상99%. 평가 데이터로 기준을 정하고 같은 데이터로 평가하므로 독립 성적이 아니다.\n\n'
report+='V1은 CN7두파일+RG3 총9129행, V2는 supervised_label_cn7 단독6736행. V2평가1379=정상1340+전체불량39. 모두 Shot/RH/LH중복정리 없이 원본 기준. 가스만의 성적이 아님. V2 학습용 별도 원본 스크립트는 공개 tree에 없어 01_train의 파일 선택만 CN7 단독으로 한 조건을 저장 배열과 대조한다.\n\n'
report+='## 배열 검증\n'+pd.DataFrame(matches).to_csv(index=False)+'\n## 결과\n'+pd.DataFrame(rows).to_csv(index=False)
(root/'reports/smartfactory_original_reproduction.md').write_text(report,encoding='utf-8')
print('COMPLETE',flush=True)


VERSIONS 3.11.8 (tags/v3.11.8:db85d51, Feb  6 2024, 22:03:32) [MSC v.1937 64 bit (AMD64)] 2.3.1+cpu 1.26.4 1.2.2
LOCAL ARRAY MATCH {'version': 'v1_combined', 'local_files': 'supervised_label_cn7.csv|moldset_labeled_cn7.csv|moldset_labeled_rg3.csv', 'shape_labels_match': True, 'max_absolute_difference': 0.0, 'exact_arrays': True}
RESULT {'version': 'v1_combined', 'method': 'published_weights_published_threshold', 'n': 1891, 'bad': 81, 'TP': 24, 'FP': 19, 'FN': 57, 'TN': 1791, 'F1': 0.3870967741935484, 'precision': 0.5581395348837209, 'recall': 0.2962962962962963, 'AP': 0.36836223793933015, 'ROC_AUC': 0.8896153059136483, 'threshold': 0.037796749882400255}
RESULT {'version': 'v1_combined', 'method': 'published_weights_original_threshold_recomputed', 'n': 1891, 'bad': 81, 'TP': 24, 'FP': 19, 'FN': 57, 'TN': 1791, 'F1': 0.3870967741935484, 'precision': 0.5581395348837209, 'recall': 0.2962962962962963, 'AP': 0.36836223793933015, 'ROC_AUC': 0.8896153059136483, 'threshold': 0.03779675021767638